In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
from loaders._gen_binary import generate_data
from sklearn.tree import DecisionTreeClassifier

In [13]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    log_loss, brier_score_loss, f1_score
)

# ---------- 1) Huấn luyện 1 cây với subspace + random search ----------
def fit_one_tree_cls(X_tr, y_tr, feat_idx, n_splits=3, seed=42, scoring="balanced_accuracy"):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    est = DecisionTreeClassifier(random_state=seed, class_weight="balanced")
    search = GridSearchCV(
        estimator=est,
        param_grid={
            "max_depth": [3, 4, 5],
            "min_samples_split": np.arange(2, 11),
            "min_samples_leaf": np.arange(1, 11),
        },
        cv=tscv, scoring=scoring, n_jobs=-1
    )
    search.fit(X_tr[:, feat_idx], y_tr)
    return search.best_estimator_, feat_idx

# ---------- 2) Xây pool ----------
def build_pool_cls(X_tr, y_tr, X_val, M=64, mtry=None, seed=42):
    rng = np.random.default_rng(seed)
    d = X_tr.shape[1]
    mtry = mtry or max(1, int(np.sqrt(d)))
    pool = []
    for m in range(M):
        feat_idx = rng.choice(d, size=mtry, replace=False)
        tree, feat_idx = fit_one_tree_cls(X_tr, y_tr, feat_idx, seed=seed+m)
        proba_val = tree.predict_proba(X_val[:, feat_idx])[:, 1]  # P(y=1)
        pool.append({"tree": tree, "feat": feat_idx, "proba_val": proba_val})
    return pool

# ---------- 3) Priors ----------
def _safe_std(x, eps=1e-12):
    s = np.std(x)
    return s if s >= eps else eps

def _safe_corr(a, b, eps=1e-12):
    sa, sb = _safe_std(a, eps), _safe_std(b, eps)
    return float(np.corrcoef((a-a.mean())/sa, (b-b.mean())/sb)[0,1])

def prior_uniform(M):
    return np.full(M, 1.0/M, dtype=float)

def prior_diversity(pool, alpha=0.5):
    """Ưu tiên các mô hình 'khác nhau' (proba ít tương quan)."""
    M = len(pool)
    H = [p["proba_val"] for p in pool]
    redund = np.zeros(M, dtype=float)
    for i in range(M):
        ci = 0.0
        for j in range(M):
            if i == j: continue
            ci += abs(_safe_corr(H[i], H[j]))
        redund[i] = ci / max(1, M-1)
    logp = -alpha * redund
    logp -= logp.max()
    p = np.exp(logp); p /= p.sum()
    return p

def prior_complexity(pool, gamma=1.0):
    """Phạt số lá để ưu tiên mô hình đơn giản."""
    leaves = np.array([p["tree"].get_n_leaves() for p in pool], dtype=float)
    if leaves.max() == leaves.min():
        return prior_uniform(len(pool))
    logp = -gamma * (leaves - leaves.min()) / (leaves.max() - leaves.min())
    logp -= logp.max()
    p = np.exp(logp); p /= p.sum()
    return p

def prior_combo(pool, alpha=0.5, gamma=0.5):
    p_div = prior_diversity(pool, alpha=alpha)
    p_cpx = prior_complexity(pool, gamma=gamma)
    p = p_div * p_cpx
    p /= p.sum()
    return p

# ---------- 4) Gibbs posterior Q* (đóng) ----------
def pac_bayes_weights(losses, prior, lam=10.0):
    """
    losses: (M,) — dùng classification loss trên validation.
            Gợi ý: log_loss hoặc 0-1 error hay Brier.
    prior:  (M,), sum=1
    lam:    temperature (lambda > 0). Lớn → nhấn mạnh mô hình tốt (loss thấp).
    """
    # Chuẩn hoá để ổn định số học
    l = losses - losses.min()
    logw = np.log(prior + 1e-16) - lam * l
    logw -= logw.max()
    w = np.exp(logw)
    w /= w.sum()
    return w

# ---------- 5) Dự đoán ensemble ----------
def predict_posterior_cls_selected(pool, weights, X, idx_sel):
    # dùng chỉ các cây được chọn
    w = weights[idx_sel]
    w = w / w.sum()
    proba = np.zeros(X.shape[0], dtype=float)
    for i, m in enumerate(idx_sel):
        p = pool[m]["tree"].predict_proba(X[:, pool[m]["feat"]])[:, 1]
        proba += w[i] * p
    y_pred = (proba >= 0.5).astype(int)
    return proba, y_pred, w  # w là trọng số sau khi chuẩn hoá trong tập chọn

# ---------- 6) Loss cho hậu nghiệm ----------
def per_model_losses(pool, y_val, loss_type="logloss"):
    y_true = y_val.astype(int)
    M = len(pool)
    losses = np.zeros(M, dtype=float)
    for i, p in enumerate(pool):
        proba = np.clip(p["proba_val"], 1e-7, 1-1e-7)
        if loss_type == "logloss":
            # ép nhãn 0/1, chỉ rõ labels để tránh lỗi khi fold thiếu lớp
            losses[i] = log_loss(y_true, np.vstack([1-proba, proba]).T, labels=[0,1])
        elif loss_type == "brier":
            losses[i] = brier_score_loss(y_true, proba)
        elif loss_type == "error":
            pred = (proba >= 0.5).astype(int)
            losses[i] = 1.0 - balanced_accuracy_score(y_true, pred)
        else:
            raise ValueError("loss_type must be one of {'logloss','brier','error'}")
    return losses

# ---------- 7) Pipeline chính ----------
def posterior_forest_classifier(
    X_train, y_train, X_val, y_val, X_test, y_test,
    *,
    M=64, mtry=None, seed=42,
    prior_kind="combo",  # {'uniform','diversity','complexity','combo'}
    alpha=0.5, gamma=0.5,
    lam=10.0, loss_type="logloss",
    select_mode="topk"  # {'topk','sample'}
):
    rng = np.random.default_rng(seed)

    # 1) Xây pool
    pool = build_pool_cls(X_train, y_train, X_val, M=M, mtry=mtry, seed=seed)

    # 2) Prior
    if prior_kind == "uniform":
        prior = prior_uniform(len(pool))
    elif prior_kind == "diversity":
        prior = prior_diversity(pool, alpha=alpha)
    elif prior_kind == "complexity":
        prior = prior_complexity(pool, gamma=gamma)
    elif prior_kind == "combo":
        prior = prior_combo(pool, alpha=alpha, gamma=gamma)
    else:
        raise ValueError("prior_kind must be in {'uniform','diversity','complexity','combo'}")

    # 3) Loss & hậu nghiệm Q*
    losses = per_model_losses(pool, y_val, loss_type=loss_type)
    weights_full = pac_bayes_weights(losses, prior, lam=lam)  # (M,)

    # 4) Chọn K = M//2 cây vào rừng
    K = max(1, len(pool)//2)
    if select_mode == "topk":
        idx_sel = np.argsort(weights_full)[-K:]
    elif select_mode == "sample":
        idx_sel = rng.choice(len(pool), size=K, replace=False, p=weights_full)
    else:
        raise ValueError("select_mode must be in {'topk','sample'}")
    idx_sel = np.sort(idx_sel)

    # 5) Dự đoán test chỉ với cây được chọn (trọng số chuẩn hoá lại trên tập chọn)
    y_proba_test, y_pred_test, weights_sel = predict_posterior_cls_selected(pool, weights_full, X_test, idx_sel)

    return balanced_accuracy_score(y_test, y_pred_test)

In [15]:
bacc_list = []
for seed in range(30):
    pack = generate_data(seed=seed, ratio=[0.6, 0.2, 0.2])
    X_train, y_train = pack["train"]
    X_val, y_val = pack["val"]
    X_test, y_test = pack["test"]

    bacc = posterior_forest_classifier(
        X_train, y_train, X_val, y_val, X_test, y_test,
        M=20, mtry=None, seed=42,
        prior_kind="combo", alpha=0.6, gamma=0.4,
        lam=10.0, loss_type="logloss"
    )
    print(f"Seed {seed}: {bacc}")
    bacc_list.append(bacc)

print(f"Mean balanced accuracy over 30 runs: {np.mean(bacc_list)}")
print(f"Std balanced accuracy over 30 runs: {np.std(bacc_list)}")

Seed 0: 0.6967336683417085
Seed 1: 0.6790992258972555
Seed 2: 0.6529318342240596
Seed 3: 0.6560612922705313
Seed 4: 0.6691919191919191
Seed 5: 0.6747009148486981
Seed 6: 0.7117713567839196
Seed 7: 0.7343216080402011
Seed 8: 0.7407999395892066
Seed 9: 0.6766095391265015
Seed 10: 0.7093698175787728
Seed 11: 0.7068896152879054
Seed 12: 0.6788570853664672
Seed 13: 0.6554026862518235
Seed 14: 0.679587374981153
Seed 15: 0.6896781493588132
Seed 16: 0.6716206030150753
Seed 17: 0.6863677536231885
Seed 18: 0.7191316146540028
Seed 19: 0.6943124368048534
Seed 20: 0.6822652044871473
Seed 21: 0.7073300015144631
Seed 22: 0.688755980861244
Seed 23: 0.6492404044469038
Seed 24: 0.6989285175310629
Seed 25: 0.6904701718907988
Seed 26: 0.669247009148487
Seed 27: 0.6917420814479638
Seed 28: 0.6726824955796918
Seed 29: 0.7167194331373437
Mean balanced accuracy over 30 runs: 0.6883606578427054
Std balanced accuracy over 30 runs: 0.02260803313944806
